# Tome 2, Chapter 5: Sylow Theorems and Finite Groups (Python companion)

Exercises first, solutions at the end. We count Sylow subgroups by brute force and check Sylow's congruences, build the groups of order $pq$, let the computer hunt for orders where a simple group might exist, compute derived series, and see Sylow theory at work in cryptography: **the Pohlig–Hellman algorithm** reduces a discrete logarithm to the Sylow subgroups of the group, which is why Diffie–Hellman must be done in a group whose order has a large prime factor.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
import random
import time
from collections import Counter
import sympy
from sympy.combinatorics import Permutation, PermutationGroup
from sympy.combinatorics.named_groups import SymmetricGroup, AlternatingGroup

## Exercises

**P5.1 (★) Counting Sylow subgroups.** For $\mathfrak S_4$, $A_5$, $\mathfrak S_5$ and $GL_2(\Z/3\Z)$ and each prime $p$ dividing the order, find all Sylow $p$-subgroups by brute force (here each is generated by two elements). Check $n_p\equiv1\pmod p$, $n_p\mid m$, and $n_p=[G:N_G(P)]$ (Theorem 5.4). Compare with Exercises 5.1 and 5.7.

In [ ]:
# Your code here


**P5.2 (★) Groups of order $pq$.** For primes $p<q<60$, list the pairs for which a non-abelian group of order $pq$ exists (Theorem 5.8). Build $\Z/q\Z\rtimes_u\Z/p\Z$ for $q=31$, $p=5$ with two different elements $u$ of order $5$ and check that the two groups have the same numbers of elements of each order.

In [ ]:
# Your code here


**P5.3 (★★) Hunting for simple groups.** For each $n\le 200$ that is not a prime power, ask whether Sylow's conditions alone ($n_p\mid m$, $n_p\equiv1\pmod p$) already force some $n_p=1$; if not, try the element count of Proposition 5.7 for the primes $p$ with $p^2\nmid n$. Which orders survive both tests? *Guess first: is $60$ among them?*

In [ ]:
# Your code here


**P5.4 (★★) Derived series.** Compute the derived series of $\mathfrak S_3$, $\mathfrak S_4$, $\mathfrak S_5$, $GL_2(\Z/3\Z)$ and $SL_2(\Z/5\Z)$ (as permutation groups on the non-zero vectors), and decide which are solvable (Exercise 5.6).

In [ ]:
# Your code here


**P5.5 (★★★) CS application: Pohlig–Hellman.** In the cyclic group $(\Z/p\Z)^*$ of order $n=p-1$, the discrete logarithm of $h$ in base $g$ is the $x$ with $g^x=h$. Generic algorithms (baby-step giant-step) need about $\sqrt n$ steps. If $n=\prod q_i^{e_i}$, the Sylow subgroup for $q_i$ is $\{y:y^{q_i^{e_i}}=1\}$; raising to the power $n/q_i^{e_i}$ projects onto it, and solving there gives $x\bmod q_i^{e_i}$, then the Chinese remainder theorem gives $x$.
1. Implement baby-step giant-step and Pohlig–Hellman.
2. Take a prime $p$ of about $63$ bits with $p-1$ a product of small primes, and a prime $p'=2q+1$ of about $40$ bits with $q$ prime (a *safe prime*). Time both algorithms.
3. Why do real protocols use a subgroup of large prime order?

In [ ]:
# Your code here


---
## Solutions

### P5.1

In [ ]:
def gl2_perm_group(p):
    vecs = [(a, b) for a in range(p) for b in range(p) if (a, b) != (0, 0)]
    idx = {v: i for i, v in enumerate(vecs)}
    def as_perm(M):
        return Permutation([idx[((M[0][0] * a + M[0][1] * b) % p, (M[1][0] * a + M[1][1] * b) % p)] for a, b in vecs])
    gens = [((1, 1), (0, 1)), ((0, 1), (1, 0)), ((2 % p if p > 2 else 1, 0), (0, 1))]
    return PermutationGroup([as_perm(M) for M in gens])

def sylow_by_brute_force(G, p):
    order = G.order()
    pa = int(p ** sympy.multiplicity(p, order))
    elems = [g for g in G.elements if g.order() == 1 or sympy.primefactors(g.order()) == [p]]
    found = set()
    for x, y in itertools.combinations_with_replacement(elems, 2):
        H = PermutationGroup([x, y])
        if H.order() == pa:
            found.add(frozenset(H.elements))
    return found, pa

groups = {'S4': SymmetricGroup(4), 'A5': AlternatingGroup(5), 'S5': SymmetricGroup(5), 'GL2(F3)': gl2_perm_group(3)}
for name, G in groups.items():
    n = int(G.order())
    for p in sympy.primefactors(n):
        syl, pa = sylow_by_brute_force(G, p)
        P = next(iter(syl))
        normaliser = [g for g in G.elements if frozenset(g * x * g**-1 for x in P) == P]
        idx_norm = n // len(normaliser)
        print(f'{name:8s} |G|={n:3d}  p={p}  |P|={pa:2d}  n_p={len(syl):3d}  n_p mod p={len(syl) % p}  m={n // pa:3d}  [G:N(P)]={idx_norm}')

Every count satisfies Sylow's two conditions, and $n_p=[G:N_G(P)]$. For $\mathfrak S_4$: $n_2=3$, $n_3=4$ (Exercise 5.1). For $A_5$: $n_2=5$, $n_3=10$, $n_5=6$: the $24$ elements of order $5$, $20$ of order $3$ and $15$ of order $2$ (in the five Klein groups, which meet trivially) fill $59$ of the $60$ elements. For $GL_2(\Z/3\Z)$, $n_3=4=p+1$, as in Exercise 5.7: one Sylow $3$-subgroup per line of the plane $(\Z/3\Z)^2$.

### P5.2

In [ ]:
ps = list(sympy.primerange(2, 60))
pairs = [(p, q) for p in ps for q in ps if p < q and (q - 1) % p == 0]
print(len(pairs), 'pairs with a non-abelian group, e.g.', pairs[:12])

def semidirect(q, p, u):
    # elements (a, b) meaning x^a y^b with y x y^-1 = x^u
    def mul(s, t):
        return ((s[0] + pow(u, s[1], q) * t[0]) % q, (s[1] + t[1]) % p)
    elems = [(a, b) for a in range(q) for b in range(p)]
    def order(g):
        k, h = 1, g
        while h != (0, 0):
            h = mul(h, g); k += 1
        return k
    abelian = all(mul(s, t) == mul(t, s) for s in elems[:40] for t in elems[:40])
    return Counter(order(g) for g in elems), abelian

q, p = 31, 5
us = [u for u in range(2, q) if pow(u, p, q) == 1]
print('elements of order 5 mod 31:', us)
for u in us[:2]:
    print('u =', u, semidirect(q, p, u))

The non-abelian groups of order $pq$ exist exactly when $p\mid q-1$: $(2,q)$ for every odd $q$ (dihedral groups), $(3,7)$, $(3,13)$, $(5,11)$, and so on. With $u=2$ and $u=4$ the two semidirect products of order $155$ have the same order statistics, $1$ element of order $1$, $30$ of order $31$ and $124$ of order $5$; Theorem 5.8 says they are in fact isomorphic, by changing the generator of $\Z/5\Z$.

### P5.3

In [ ]:
def sylow_options(n, p):
    m = n
    while m % p == 0:
        m //= p
    return [d for d in sympy.divisors(m) if d % p == 1]

survivors = []
for n in range(2, 201):
    fac = sympy.factorint(n)
    if len(fac) == 1:
        continue                                   # p-groups have a non-trivial centre: not simple unless prime
    opts = {p: [d for d in sylow_options(n, p) if d > 1] for p in fac}
    if any(not o for o in opts.values()):
        continue                                   # some n_p is forced to be 1
    # element count: for p with p || n, the n_p subgroups of order p give n_p (p - 1) elements of order p
    small = [p for p, e in fac.items() if e == 1]
    min_count = sum(min(opts[p]) * (p - 1) for p in small)
    if min_count + 1 > n:
        continue
    survivors.append(n)
print(survivors)

Only $20$ orders up to $200$ survive these two quick tests: $12$, $24$, $36$, $48$, $56$, $60$, $72$, $80$, $90$, $96$, $108$, $112$, $120$, $132$, $144$, $150$, $160$, $168$, $180$ and $192$. Odd orders all fail, and so do the orders $pq$ and most others. For most of them an extra argument kills simplicity: $12$, $24$, $36$, $48$ by letting $G$ act on few Sylow subgroups (Exercise 5.3(c)), $56$ by counting the elements of order $7$ *and* noticing the leftovers form the Sylow $2$-subgroup (Exercise 5.3(a)). The two genuine simple groups below $200$ are $A_5$ of order $60$ and $GL_3(\Z/2\Z)$ of order $168$. The computer is a good tool to find candidates; the arguments remain human.

### P5.4

In [ ]:
def sl2_perm_group(p):
    vecs = [(a, b) for a in range(p) for b in range(p) if (a, b) != (0, 0)]
    idx = {v: i for i, v in enumerate(vecs)}
    def as_perm(M):
        return Permutation([idx[((M[0][0] * a + M[0][1] * b) % p, (M[1][0] * a + M[1][1] * b) % p)] for a, b in vecs])
    return PermutationGroup([as_perm(((1, 1), (0, 1))), as_perm(((1, 0), (1, 1)))])

for name, G in [('S3', SymmetricGroup(3)), ('S4', SymmetricGroup(4)), ('S5', SymmetricGroup(5)),
                ('GL2(F3)', gl2_perm_group(3)), ('SL2(F5)', sl2_perm_group(5))]:
    print(f'{name:8s} derived series orders {[H.order() for H in G.derived_series()]}  solvable: {G.is_solvable}')

$\mathfrak S_4\supset A_4\supset V\supset\{e\}$ ($24,12,4,1$) and $\mathfrak S_5\supset A_5$ which stops at $60$, as in Exercise 5.6. $GL_2(\Z/3\Z)$ (order $48$) is solvable, with derived series of orders $48,24,8,2,1$. $SL_2(\Z/5\Z)$ (order $120$) is not solvable: it is perfect, $G'=G$. Its quotient by $\{\pm I\}$ is a simple group of order $60$, isomorphic to $A_5$, and $SL_2(\Z/5\Z)$ is the binary icosahedral group of Chapter 12.

### P5.5

In [ ]:
def bsgs(g, h, n, p):
    # solve g^x = h in a group of order n inside (Z/p)^*
    m = math.isqrt(n) + 1
    table = {}
    e = 1
    for j in range(m):
        table.setdefault(e, j); e = e * g % p
    factor = pow(g, -m, p)
    y = h
    for i in range(m + 1):
        if y in table:
            return (i * m + table[y]) % n
        y = y * factor % p
    return None

def pohlig_hellman(g, h, p):
    n = p - 1
    residues, moduli = [], []
    for q, e in sympy.factorint(n).items():
        qe = q ** e
        g1, h1 = pow(g, n // qe, p), pow(h, n // qe, p)      # project onto the Sylow q-subgroup
        # solve g1^x = h1 digit by digit in base q (each digit is a log in a group of order q)
        gq = pow(g1, qe // q, p)                             # element of order q
        x = 0
        for k in range(e):
            hk = pow(h1 * pow(g1, -x, p) % p, qe // q ** (k + 1), p)
            d = bsgs(gq, hk, q, p)
            x += d * q ** k
        residues.append(x); moduli.append(qe)
    return sympy.ntheory.modular.crt(moduli, residues)[0]

random.seed(5)
# a 64-bit prime with smooth p - 1
while True:
    n = 2
    while n.bit_length() < 63:
        n *= random.choice([3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47])
    if sympy.isprime(n + 1):
        p = n + 1; break
g = sympy.primitive_root(p)
x = random.randrange(p - 1); h = pow(g, x, p)
t = time.perf_counter(); y = pohlig_hellman(g, h, p); t1 = time.perf_counter() - t
print(f'p = {p} ({p.bit_length()} bits), p-1 = {sympy.factorint(p - 1)}')
print(f'Pohlig-Hellman: correct={pow(g, y, p) == h}  time {t1 * 1000:.1f} ms')

# a 40-bit safe prime p' = 2q + 1
while True:
    q = sympy.randprime(2**38, 2**39)
    if sympy.isprime(2 * q + 1):
        ps = 2 * q + 1; break
g2 = sympy.primitive_root(ps)
x2 = random.randrange(ps - 1); h2 = pow(g2, x2, ps)
t = time.perf_counter(); y2 = pohlig_hellman(g2, h2, ps); t2 = time.perf_counter() - t
print(f'safe prime {ps} ({ps.bit_length()} bits): correct={pow(g2, y2, ps) == h2}  time {t2 * 1000:.0f} ms')

Pohlig–Hellman solves the $64$-bit smooth case in milliseconds, because the hardest step is a logarithm in a group of order at most $47$. On the $40$-bit safe prime, it has to run baby-step giant-step in the Sylow subgroup of order $q\approx2^{39}$, about $2^{20}$ steps, and is already hundreds of times slower than on a prime $2^{23}$ times larger. In general the cost is about $\sqrt{q_{\max}}$, where $q_{\max}$ is the largest prime factor of the group order: **a discrete logarithm is only as hard as in the largest Sylow subgroup.**

That is why Diffie–Hellman and DSA work in a subgroup of prime order $q$ (with $q$ of $256$ bits or more), and why elliptic-curve standards choose curves whose number of points is a large prime times a tiny cofactor. Sylow's theorem, through the Chinese remainder theorem, is the reason the group order must be checked before a group is trusted with a secret.